In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Konfiguracja stylu
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Wczytanie danych
with open('long_training_vit_best_150ep.json', 'r', encoding='utf-8') as f:
    long_train_data = json.load(f)

with open('porownanie_wynikow.json', 'r', encoding='utf-8') as f:
    main_data = json.load(f)

cifar10_long = long_train_data['cifar10']
cifar10_main = main_data['cifar10']

print('Dane załadowane pomyślnie!')

## 1. Podsumowanie Wyniku - 150 Epok

In [ ]:
print('\n' + '='*80)
print('🎯 ViT_Best: 150 Epochs na CIFAR-10')
print('='*80)
print(f'\n📊 WYNIKI:')
print(f'  Test Accuracy: {cifar10_long["test_acc"]:.2f}%')
print(f'  Train Accuracy: {cifar10_long["train_acc"]:.2f}%')
print(f'  Best Test Accuracy (Epoch {cifar10_long["best_epoch"]}): {cifar10_long["best_test_acc"]:.2f}%')
print(f'\n⚙️ PARAMETRY:')
print(f'  Liczba parametrów: {cifar10_long["num_params"]/1e6:.2f}M')
print(f'  Patch size: {cifar10_long["config"]["patch_size"]}')
print(f'  Embedding dim: {cifar10_long["config"]["embed_dim"]}')
print(f'  Number of heads: {cifar10_long["config"]["num_heads"]}')
print(f'  Transformer blocks: {cifar10_long["config"]["Nx"]}')
print(f'  Learning rate: {cifar10_long["config"]["lr"]}')
print(f'\n⏱️ CZAS:')
print(f'  Czas treningu: {cifar10_long["train_time_s"]/3600:.2f} godzin')
print(f'  Łącznie sekund: {cifar10_long["train_time_s"]:.0f}s')
print(f'  Średnio na epokę: {cifar10_long["time_per_epoch"]:.2f}s')
print('\n' + '='*80)

## 2. Krzywa Treningu: Dokładność

In [ ]:
# Przygotowanie danych
history_150 = cifar10_long['history']
epochs_150 = [h['epoch'] for h in history_150]
train_acc_150 = [h['train_acc'] for h in history_150]
test_acc_150 = [h['test_acc'] for h in history_150]
loss_150 = [h['loss'] for h in history_150]

# Znajdź najlepszą docładność
best_idx = np.argmax(test_acc_150)
best_epoch = epochs_150[best_idx]
best_acc = test_acc_150[best_idx]

# Wykres dokładności
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(epochs_150, train_acc_150, 'o-', label='Train Accuracy', color='#2E86AB', linewidth=2.5, markersize=5)
ax.plot(epochs_150, test_acc_150, 's-', label='Test Accuracy', color='#A23B72', linewidth=2.5, markersize=5)

# Zaznacz najlepszy wynik
ax.scatter([best_epoch], [best_acc], s=300, color='gold', edgecolor='darkred', linewidth=2, zorder=5, label=f'Best: {best_acc:.2f}% (Ep. {best_epoch})')

ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('ViT_Best: 150 Epochs - Test Accuracy Progression', fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.grid(alpha=0.3, linestyle='--')
ax.set_xlim([0, max(epochs_150) + 5])
ax.set_ylim([0, 100])

plt.tight_layout()
plt.savefig('bonus_01_accuracy_150epochs.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'\n✨ Najlepszy wynik: {best_acc:.2f}% na Epoch {best_epoch}')

## 3. Krzywa Treningu: Loss

In [ ]:
# Wykres loss
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(epochs_150, loss_150, '^-', label='Training Loss', color='#F18F01', linewidth=2.5, markersize=5)

# Zaznacz punkt z najlepszą dokładnością
ax.scatter([best_epoch], [loss_150[best_idx]], s=300, color='gold', edgecolor='darkred', linewidth=2, zorder=5)

ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
ax.set_ylabel('Loss', fontsize=12, fontweight='bold')
ax.set_title('ViT_Best: 150 Epochs - Training Loss', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3, linestyle='--')
ax.set_xlim([0, max(epochs_150) + 5])

plt.tight_layout()
plt.savefig('bonus_02_loss_150epochs.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Porównanie: 10 epok vs 150 epok

In [ ]:
# Wyciągnij dane z głównych wyników dla ViT_Best (10 epok)
vit_best_10ep = None
for item in cifar10_main:
    if item.get('experiment') == 'ViT_Best':
        vit_best_10ep = item
        break

# Stwórz porównanie
comparison_df = pd.DataFrame({
    'Configuration': ['ViT_Best (10 ep)', 'ViT_Best (150 ep)'],
    'Test Accuracy (%)': [vit_best_10ep['test_acc'], cifar10_long['test_acc']],
    'Best Accuracy (%)': [vit_best_10ep['test_acc'], cifar10_long['best_test_acc']],
    'Training Time (s)': [vit_best_10ep['train_time_s'], cifar10_long['train_time_s']],
    'Epochs': [10, 150]
})

print('\n📊 PORÓWNANIE: 10 epok vs 150 epok')
print('='*80)
print(comparison_df.to_string(index=False))

# Wykresy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Dokładność
x_pos = np.arange(len(comparison_df))
colors = ['#4ECDC4', '#1B998B']

axes[0].bar(x_pos, comparison_df['Test Accuracy (%)'], color=colors, alpha=0.8, edgecolor='black', linewidth=2)
axes[0].set_ylabel('Test Accuracy (%)', fontsize=11, fontweight='bold')
axes[0].set_title('Porównanie: Test Accuracy', fontsize=12, fontweight='bold')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(comparison_df['Configuration'])
axes[0].set_ylim([70, 82])

for i, v in enumerate(comparison_df['Test Accuracy (%)']):
    axes[0].text(i, v + 0.3, f'{v:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)

axes[0].grid(axis='y', alpha=0.3)

# Czas treningu
time_hours = comparison_df['Training Time (s)'] / 3600
axes[1].bar(x_pos, time_hours, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
axes[1].set_ylabel('Training Time (hours)', fontsize=11, fontweight='bold')
axes[1].set_title('Porównanie: Czas Treningu', fontsize=12, fontweight='bold')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(comparison_df['Configuration'])

for i, (v_s, v_h) in enumerate(zip(comparison_df['Training Time (s)'], time_hours)):
    axes[1].text(i, v_h + 0.1, f'{v_h:.2f}h\n({v_s:.0f}s)', ha='center', va='bottom', fontweight='bold', fontsize=10)

axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('bonus_03_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# Statystyka
acc_improvement = cifar10_long['test_acc'] - vit_best_10ep['test_acc']
time_ratio = cifar10_long['train_time_s'] / vit_best_10ep['train_time_s']

print(f'\n\n💡 WNIOSKI:')
print(f'  Wzrost dokładności: +{acc_improvement:.2f}% (z {vit_best_10ep["test_acc"]:.2f}% do {cifar10_long["test_acc"]:.2f}%)')
print(f'  Wzrost czasu: {time_ratio:.1f}x więcej czasu')
print(f'  Efektywność: +{acc_improvement/time_ratio:.4f}% dokładności na godzinę dodatkowego treningu')

## 5. Krzywe Treningu: Wszystkie metryki

In [ ]:
# Kombinowany wykres
fig, ax1 = plt.subplots(figsize=(13, 6))

# Lewy Y-axis: Dokładność
color1 = '#2E86AB'
ax1.set_xlabel('Epoch', fontsize=12, fontweight='bold')
ax1.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold', color=color1)
ax1.plot(epochs_150, train_acc_150, 'o-', label='Train Accuracy', color=color1, linewidth=2.5, markersize=4, alpha=0.7)
ax1.plot(epochs_150, test_acc_150, 's-', label='Test Accuracy', color='#A23B72', linewidth=2.5, markersize=4)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.grid(alpha=0.3, linestyle='--')

# Prawy Y-axis: Loss
ax2 = ax1.twinx()
color2 = '#F18F01'
ax2.set_ylabel('Loss', fontsize=12, fontweight='bold', color=color2)
ax2.plot(epochs_150, loss_150, '^-', label='Loss', color=color2, linewidth=2.5, markersize=4)
ax2.tick_params(axis='y', labelcolor=color2)

ax1.set_title('ViT_Best: 150 Epochs - Pełny Przebieg Treningu', fontsize=13, fontweight='bold')
ax1.set_xlim([0, max(epochs_150) + 5])

# Legenda
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right', fontsize=11)

plt.tight_layout()
plt.savefig('bonus_04_full_training_curve.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Milestones: Postęp Treningu

In [ ]:
# Wybierz milestones
milestones = [10, 20, 30, 50, 75, 100, 125, 150]
milestone_data = []

for ep in milestones:
    for h in history_150:
        if h['epoch'] == ep:
            milestone_data.append({
                'Epoch': ep,
                'Train Acc': h['train_acc'],
                'Test Acc': h['test_acc'],
                'Loss': h['loss']
            })
            break

milestone_df = pd.DataFrame(milestone_data)

print('\n📍 MILESTONES - POSTĘP TRENINGU')
print('='*70)
print(milestone_df.to_string(index=False))

# Wykres milestones
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(milestone_df))
width = 0.35

bars1 = ax.bar(x - width/2, milestone_df['Train Acc'], width, label='Train Accuracy', color='#2E86AB', alpha=0.8, edgecolor='black')
bars2 = ax.bar(x + width/2, milestone_df['Test Acc'], width, label='Test Accuracy', color='#A23B72', alpha=0.8, edgecolor='black')

ax.set_xlabel('Epoch Milestone', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('ViT_Best: Milestones - Accuracy Progression', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'Ep {int(e)}' for e in milestone_df['Epoch']], rotation=45)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim([0, 100])

plt.tight_layout()
plt.savefig('bonus_05_milestones.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Wnioski - Long Training

In [ ]:
print('\n' + '='*80)
print('🎯 WNIOSKI - TRENING Z 150 EPOKAMI')
print('='*80)

print(f'\n1️⃣ NAJWYŻSZY WYNIK:')
print(f'   • Best Test Accuracy: {cifar10_long["best_test_acc"]:.2f}% (Epoch {cifar10_long["best_epoch"]})')
print(f'   • Final Test Accuracy: {cifar10_long["test_acc"]:.2f}%')
print(f'   • Train Accuracy: {cifar10_long["train_acc"]:.2f}% (overfitting: {cifar10_long["train_acc"] - cifar10_long["test_acc"]:.2f}%)')

print(f'\n2️⃣ PORÓWNANIE Z 10 EPOKAMI:')
print(f'   • Wzrost dokładności: +{acc_improvement:.2f}%')
print(f'   • Dodatkowy czas treningu: {(cifar10_long["train_time_s"] - vit_best_10ep["train_time_s"])/3600:.2f} godzin')
print(f'   • Zysk dokładności na godzinę: {acc_improvement/(cifar10_long["train_time_s"] - vit_best_10ep["train_time_s"])*3600:.4f}%')

print(f'\n3️⃣ DYNAMIKA TRENINGU:')
first_50 = [h['test_acc'] for h in history_150 if h['epoch'] == 50][0]
first_100 = [h['test_acc'] for h in history_150 if h['epoch'] == 100][0]
print(f'   • Accuracy po 50 epokach: {first_50:.2f}%')
print(f'   • Accuracy po 100 epokach: {first_100:.2f}%')
print(f'   • Accuracy po 150 epokach: {cifar10_long["test_acc"]:.2f}%')
print(f'   • Wzrost w drugiej połowie: +{cifar10_long["test_acc"] - first_100:.2f}%')

print(f'\n4️⃣ REKOMENDACJE:')
print(f'   • Długi trening (150 epok) wart jest dla finalnego modelu')
print(f'   • Najlepszy wynik ({cifar10_long["best_test_acc"]:.2f}%) osiągnięty na epoch {cifar10_long["best_epoch"]}')
print(f'   • Model nadal uczy się do końca (brak plateau)')
print(f'   • Wymagany czas: ~{cifar10_long["train_time_s"]/3600:.1f} godzin na GPU')

print('\n' + '='*80)